### Save all ranking history

In [15]:
import json
import pandas as pd
import os
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option("display.max_rows", None)

# --> Import functions from "process_trajectory_data" script
import sys
sys.path.append('../src')

from process import get_ranking_info
from search_utils import get_player_by_name_df, get_all_players_info_df, get_player_ranking_history, get_all_players_info_df


In [16]:
base_directory = r'../all_json/ranking_history'

directory_list = ['atp', 'wta']

processed_data_folder = 'ranking_data_csvs'

for directory in directory_list:
    dir_name = os.path.join(base_directory, directory)
    
    if not os.path.exists(dir_name):
        print(f"Directory '{dir_name}' does not exist. Skipping...")
        continue
    
    csv_save_path = os.path.join(processed_data_folder, directory)
    os.makedirs(csv_save_path, exist_ok=True) 
        
    for filename in os.listdir(dir_name):
        if filename.endswith(".json"):
            print(f"Processing {filename}")
            with open(os.path.join(dir_name, filename)) as file_name:
                ranking_data_json = json.load(file_name)
                
                ranking_data = get_ranking_info(ranking_data_json)
                
                save_data_filename = f"{filename[:-5]}.csv"
                save_data_path = os.path.join(csv_save_path, save_data_filename)
                
                print("Saving to:", save_data_path)
                
                ranking_data.to_csv(save_data_path, index=False)


Processing atp_A0E2_player_ranking_history.json
Saving to: ranking_data_csvs\atp\atp_A0E2_player_ranking_history.csv
Processing atp_A0FC_player_ranking_history.json
Saving to: ranking_data_csvs\atp\atp_A0FC_player_ranking_history.csv
Processing atp_A0GC_player_ranking_history.json
Saving to: ranking_data_csvs\atp\atp_A0GC_player_ranking_history.csv
Processing atp_A596_player_ranking_history.json
Saving to: ranking_data_csvs\atp\atp_A596_player_ranking_history.csv
Processing atp_A678_player_ranking_history.json
Saving to: ranking_data_csvs\atp\atp_A678_player_ranking_history.csv
Processing atp_A829_player_ranking_history.json
Saving to: ranking_data_csvs\atp\atp_A829_player_ranking_history.csv
Processing atp_AA27_player_ranking_history.json
Saving to: ranking_data_csvs\atp\atp_AA27_player_ranking_history.csv
Processing atp_AE14_player_ranking_history.json
Saving to: ranking_data_csvs\atp\atp_AE14_player_ranking_history.csv
Processing atp_AG37_player_ranking_history.json
Saving to: ranki

### Add ranking stuff to matches catalogue

In [17]:
catalogue_path = "players_matches_catalogue/catalogue_all_matches_available.csv"
catalogue_df = pd.read_csv(catalogue_path)

In [18]:
players_df= get_all_players_info_df('', 'atp')

players_df_rg = get_all_players_info_df('roland_garros', 'atp')
players_df_rg = get_all_players_info_df('australian_open', 'atp')

In [20]:
players_df.head()

,player_name,player_id_ao,league,player_id_rg,country,player_id_atp
1,A.BALAZS,ATPBD80,atp,15374,HUN,BD80
3,A.BEDENE,ATPBH09,atp,19666,SLO,BH09
7,A.BOLT,ATPBI81,atp,0,NaN,BI81
9,A.BUBLIK,ATPBK92,atp,29098,KAZ,BK92
10,A.CAZAUX,ATPC0H0,atp,43952,FRA,C0H0


In [25]:
players_matches_catalogue_folder = 'players_matches_catalogue'

catalogue_csv_path = os.path.join(players_matches_catalogue_folder, 'catalogue_all_matches_available.csv')


### Add rolling ranking and points to matches in catalogue

In [26]:
league = 'atp'
catalogue = catalogue_df[catalogue_df['league'] == league]
for index, row in catalogue.iterrows():
    tournament = row['tournament']
    player1_id = row['player1_id']
    player2_id = row['player2_id']
    
    if tournament == 'australian_open':
        player1_info = players_df[players_df['player_id_ao'] == player1_id]
        player2_info = players_df[players_df['player_id_ao'] == player2_id]
    elif tournament == 'roland_garros':
        player1_info = players_df[players_df['player_id_rg'] == int(player1_id)]
        player2_info = players_df[players_df['player_id_rg'] == int(player2_id)]
    else:
        continue
    
    if not player1_info.empty and not player2_info.empty:
        player1_id_atp = str(player1_info['player_id_atp'].iloc[0])
        player2_id_atp = str(player2_info['player_id_atp'].iloc[0])
        
        player1_ranking_history = get_player_ranking_history(player1_id_atp, league)
        player2_ranking_history = get_player_ranking_history(player2_id_atp, league)
        
        player1_ranking_history['rank_date'] = pd.to_datetime(player1_ranking_history['rank_date'], format='%d-%m-%Y')
        player2_ranking_history['rank_date'] = pd.to_datetime(player2_ranking_history['rank_date'], format='%d-%m-%Y')
        
        # Get the ranking for the year of the match
        match_year = row['year']
        player1_ranking = player1_ranking_history[player1_ranking_history['rank_date'].dt.year == match_year]['singles_roll_rank'].values
        player2_ranking = player2_ranking_history[player2_ranking_history['rank_date'].dt.year == match_year]['singles_roll_rank'].values
        
        player1_points = player1_ranking_history[player1_ranking_history['rank_date'].dt.year == match_year]['singles_roll_points'].values
        player2_points = player2_ranking_history[player2_ranking_history['rank_date'].dt.year == match_year]['singles_roll_points'].values
        
        if player1_ranking.size > 0:
            catalogue_df.at[index, 'player1_ranking'] = int(player1_ranking[0])
        if player2_ranking.size > 0:
            catalogue_df.at[index, 'player2_ranking'] = int(player2_ranking[0])
        
        if player1_points.size > 0:
            catalogue_df.at[index, 'player1_points'] = int(player1_points[0])
        if player2_points.size > 0:
            catalogue_df.at[index, 'player2_points'] = int(player2_points[0])
            
            
catalogue_df['player1_ranking'] = catalogue_df['player1_ranking'].fillna(0)
catalogue_df['player2_ranking'] = catalogue_df['player2_ranking'].fillna(0)
catalogue_df['player1_ranking'] = catalogue_df['player1_ranking'].astype(int)
catalogue_df['player2_ranking'] = catalogue_df['player2_ranking'].astype(int)

catalogue_df['player1_points'] = catalogue_df['player1_points'].fillna(0)
catalogue_df['player2_points'] = catalogue_df['player2_points'].fillna(0)
catalogue_df['player1_points'] = catalogue_df['player1_points'].astype(int)
catalogue_df['player2_points'] = catalogue_df['player2_points'].astype(int)


catalogue_df['player1_seed'] = catalogue_df['player1_seed'].fillna(0)
catalogue_df['player2_seed'] = catalogue_df['player2_seed'].fillna(0)

catalogue_df['player1_seed'] = catalogue_df['player1_seed'].astype(int)
catalogue_df['player2_seed'] = catalogue_df['player2_seed'].astype(int)

catalogue_df.to_csv(catalogue_csv_path, index=False)

print("Updated catalogue saved to " + catalogue_csv_path)


Updated catalogue saved to players_matches_catalogue\catalogue_all_matches_available.csv


### Add avg ranking to players.csv

In [29]:
players_df = pd.read_csv('players.csv')
catalogue_df = pd.read_csv('players_matches_catalogue/catalogue_all_matches_available.csv')

player_rankings = {}

for index, row in catalogue_df.iterrows():
    player1_id = str(row['player1_id'])
    player2_id = str(row['player2_id'])
    player1_ranking = row['player1_ranking']
    player2_ranking = row['player2_ranking']
    
    if player1_id in player_rankings:
        player_rankings[player1_id].append(player1_ranking)
    else:
        player_rankings[player1_id] = [player1_ranking]
        
    if player2_id in player_rankings:
        player_rankings[player2_id].append(player2_ranking)
    else:
        player_rankings[player2_id] = [player2_ranking]

def calculate_avg_ranking(rankings):
    if not rankings:
        return None
    return sum(rankings) / len(rankings)

avg_rankings = []
if player1_id == '11713' or player2_id == '11713':
    print("player1_info")
    print("player2_info")

for index, row in players_df.iterrows():
    player_id_ao = row['player_id_ao']
    player_id_rg = str(row['player_id_rg'])
    player_id_atp = row['player_id_atp']
    
    all_rankings = []
    if player_id_ao in player_rankings:
        all_rankings.extend(player_rankings[player_id_ao])
    if player_id_rg in player_rankings:
        all_rankings.extend(player_rankings[str(player_id_rg)])
    if player_id_atp in player_rankings:
        all_rankings.extend(player_rankings[player_id_atp])
    
    avg_ranking = calculate_avg_ranking(all_rankings)
    print()
    avg_rankings.append(avg_ranking)

players_df['avg_ranking'] = avg_rankings
players_df['avg_ranking'] = players_df['avg_ranking'].fillna(0)
players_df['avg_ranking'] = players_df['avg_ranking'].round(2)

players_df.to_csv('players.csv', index=False)

### Add avg rolling points to players.csv

In [32]:
players_df = pd.read_csv('players.csv')
catalogue_df = pd.read_csv('players_matches_catalogue/catalogue_all_matches_available.csv')

player_points = {}

for index, row in catalogue_df.iterrows():
    player1_id = str(row['player1_id'])
    player2_id = str(row['player2_id'])
    player1_point = row['player1_points']
    player2_point = row['player2_points']
    
    if player1_id in player_points:
        player_points[player1_id].append(player1_point)
    else:
        player_points[player1_id] = [player1_point]
        
    if player2_id in player_points:
        player_points[player2_id].append(player2_point)
    else:
        player_points[player2_id] = [player2_point]

def calculate_avg_point(points):
    if not points:
        return None
    return sum(points) / len(points)

avg_points = []
for index, row in players_df.iterrows():
    player_id_ao = str(row['player_id_ao'])
    player_id_rg = str(row['player_id_rg'])
    player_id_atp = row['player_id_atp']
    
    all_points = []
    if player_id_ao in player_points:
        all_points.extend(player_points[player_id_ao])
    if player_id_rg in player_points:
        all_points.extend(player_points[str(player_id_rg)])
    if player_id_atp in player_points:
        all_points.extend(player_points[player_id_atp])
    
    avg_point = calculate_avg_point(all_points)
    avg_points.append(avg_point)
    if player_id_rg == str(11713):
        print("player_points", player_points[str(player_id_rg)])
        print("avg_point", all_points)

players_df['avg_points'] = avg_points
players_df['avg_points'] = players_df['avg_points'].fillna(0)
players_df['avg_points'] = players_df['avg_points'].astype(int)


players_df.to_csv('players.csv', index=False)

player_points [460, 460]
avg_point [460, 460]
